In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/races.csv'
table_name = f"{catalog_name}.{bronze_schema}.races"

#Ingest races csv file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns
-        Source File
-        Ingestion Timestamp
3. Write to bronze delta table

In [0]:
from pyspark.sql.types import StructType,StructField,StringType, DateType, IntegerType
races_schema = StructType([
    StructField('season',IntegerType()),
    StructField('round',IntegerType()),
    StructField('url',StringType()),
    StructField('raceName',StringType()),
    StructField('date',DateType()),
    StructField('circuitId',StringType())]
)

In [0]:
races_df = (spark
.read
.format("csv")
.option("header","true")
.schema(races_schema)
.option('mode','FAILFAST')
.load(source_file))

In [0]:

races_final_df =  add_ingestion_metadata(races_df)


In [0]:
display(races_final_df)

In [0]:
write_to_bronze(races_final_df,table_name,batch_id=v_batch_id)

In [0]:
display(spark.table(table_name))